In [1]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = "../data"
VAL_DIR = os.path.join(BASE_DIR, "Validation")
TEST_DIR = os.path.join(BASE_DIR, "Test")
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

normalization_layer = tf.keras.layers.Rescaling(1./255)

val_ds_nor = val_ds.map(lambda images, labels: (normalization_layer(images), labels))
test_ds_nor = test_ds.map(lambda images, labels: (normalization_layer(images), labels))

AUTOTUNE = tf.data.AUTOTUNE
val_ds_final = val_ds_nor.cache().prefetch(buffer_size=AUTOTUNE)
test_ds_final = test_ds_nor.cache().prefetch(buffer_size=AUTOTUNE)

print('dataset loaded')

KeyboardInterrupt: 

In [ ]:
simple_cnn = load_model("../model/mask_complex_CNN_01.keras")
complex_cnn = load_model("../model/mask_complex_CNN_01.keras")
transfer = load_model("../model/mask_complex_CNN_01.keras")

print('models loaded')

In [ ]:
#simple cnn
print("Evaluating simple CNN on test set...")

y_pred_probs = simple_cnn.predict(test_ds_final, verbose=1)
y_true = np.concatenate([labels for _, labels in test_ds_final], axis=0)

y_true = y_true.astype(int).flatten()
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

conf_matrix = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(conf_matrix)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['WithMask', 'WithoutMask']))

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['WithMask', 'WithoutMask'], 
            yticklabels=['WithMask', 'WithoutMask'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Simple CNN', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#complex cnn
print("Evaluating complex CNN on test set...")
test_results = complex_cnn.evaluate(test_ds_final, verbose=1)

y_pred_probs = complex_cnn.predict(test_ds_final, verbose=1)
y_true = np.concatenate([labels for _, labels in test_ds_final], axis=0)

y_true = y_true.astype(int).flatten()
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

f1 = f1_score(y_true, y_pred) #i forgot to put F1 score to trained model so i have to include it here
conf_matrix = confusion_matrix(y_true, y_pred)

print(f"F1-Score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['WithMask', 'WithoutMask']))

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['WithMask', 'WithoutMask'], 
            yticklabels=['WithMask', 'WithoutMask'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Complex CNN', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#transfer
print("Evaluating transfer learning on test set...")

y_pred_probs = transfer.predict(test_ds_final, verbose=1)
y_true = np.concatenate([labels for _, labels in test_ds_final], axis=0)

y_true = y_true.astype(int).flatten()
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

conf_matrix = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(conf_matrix)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['WithMask', 'WithoutMask']))

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Reds', 
            xticklabels=['WithMask', 'WithoutMask'], 
            yticklabels=['WithMask', 'WithoutMask'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Transfer learning', fontsize=14)
plt.tight_layout()
plt.show()